# Mini-FORESIGHT — Step 4: Data Cleaning

In the previous notebook we **understood** the data. Now we **clean** it so it is ready for analysis and forecasting.

Cleaning tasks we will do:
- Convert the `date` column from text to a real **datetime** type
- Standardize text columns (strip whitespace, uppercase SKU IDs)
- Check for and remove duplicate rows
- Re-check missing values after cleaning
- Validate the inventory stock formula
- Save the cleaned datasets to `data/processed/`

We will **NOT**:
- Build features
- Train a machine learning model
- Create a Streamlit app

## 1. Import Libraries

We use **pandas** for cleaning and **pathlib** to build file paths safely.

In [ ]:
import pandas as pd
from pathlib import Path

print('pandas version:', pd.__version__)

## 2. Load the Raw Datasets

We read the same three files from `data/raw/` that we explored in the data understanding notebook.

In [ ]:
raw_dir = Path('data/raw')

sales_daily = pd.read_csv(raw_dir / 'sales_daily.csv')
sku_master = pd.read_csv(raw_dir / 'sku_master.csv')
inventory_snapshots = pd.read_csv(raw_dir / 'inventory_snapshots.csv')

print('Loaded raw files:')
print('  sales_daily         ->', sales_daily.shape)
print('  sku_master          ->', sku_master.shape)
print('  inventory_snapshots ->', inventory_snapshots.shape)

## 3. Clean the Date Column

In the raw files, `date` was read as **text** (`str`).
We convert it to a real **datetime** type with `pd.to_datetime()` so pandas can sort, filter and compare dates correctly.

In [ ]:
sales_daily['date'] = pd.to_datetime(sales_daily['date'])
inventory_snapshots['date'] = pd.to_datetime(inventory_snapshots['date'])

print('sales_daily date dtype:', sales_daily['date'].dtype)
print('inventory_snapshots date dtype:', inventory_snapshots['date'].dtype)
print()
print('First 3 dates in sales_daily:')
print(sales_daily['date'].head(3).tolist())

## 4. Clean Text Columns

Text columns can contain accidental spaces or inconsistent casing.
We standardize them:
- `sku_id` → strip whitespace and convert to **uppercase**
- `product_name` and `category` → strip whitespace and convert to **title case**

In [ ]:
# Helper function: clean a text column
def clean_text(series, case='upper'):
    series = series.astype(str).str.strip()
    if case == 'upper':
        return series.str.upper()
    elif case == 'title':
        return series.str.title()
    return series

# Apply to sales_daily
sales_daily['sku_id'] = clean_text(sales_daily['sku_id'], case='upper')

# Apply to sku_master
sku_master['sku_id'] = clean_text(sku_master['sku_id'], case='upper')
sku_master['product_name'] = clean_text(sku_master['product_name'], case='title')
sku_master['category'] = clean_text(sku_master['category'], case='title')

# Apply to inventory_snapshots
inventory_snapshots['sku_id'] = clean_text(inventory_snapshots['sku_id'], case='upper')

print('Cleaned text columns:')
print(sku_master)

## 5. Check for Duplicate Rows

Duplicate rows would double-count sales or stock. We check with `.duplicated()` and remove any with `.drop_duplicates()`.

In [ ]:
print('Duplicate rows before cleaning:')
print('  sales_daily        :', sales_daily.duplicated().sum())
print('  sku_master         :', sku_master.duplicated().sum())
print('  inventory_snapshots:', inventory_snapshots.duplicated().sum())

# Remove duplicates (keeps the first occurrence)
sales_daily = sales_daily.drop_duplicates()
sku_master = sku_master.drop_duplicates()
inventory_snapshots = inventory_snapshots.drop_duplicates()

print()
print('Duplicate rows after cleaning:')
print('  sales_daily        :', sales_daily.duplicated().sum())
print('  sku_master         :', sku_master.duplicated().sum())
print('  inventory_snapshots:', inventory_snapshots.duplicated().sum())

## 6. Re-check Missing Values

After cleaning, we confirm no missing values remain in any dataset.

In [ ]:
print('Missing values after cleaning:')
print()
print('--- sales_daily ---')
print(sales_daily.isna().sum())
print()
print('--- sku_master ---')
print(sku_master.isna().sum())
print()
print('--- inventory_snapshots ---')
print(inventory_snapshots.isna().sum())

## 7. Validate the Inventory Stock Formula

A key data-quality rule: `closing_stock = opening_stock + units_received - units_sold`.
We verify this holds for every row after cleaning.

In [ ]:
calculated_closing = (
    inventory_snapshots['opening_stock']
    + inventory_snapshots['units_received']
    - inventory_snapshots['units_sold']
)

formula_ok = (calculated_closing == inventory_snapshots['closing_stock']).all()
print('Stock formula holds for every row:', formula_ok)

if not formula_ok:
    bad_rows = inventory_snapshots[calculated_closing != inventory_snapshots['closing_stock']]
    print('Rows where the formula fails:')
    print(bad_rows)

## 8. Verify SKU Consistency Across Datasets

Every SKU in `sales_daily` and `inventory_snapshots` must exist in `sku_master`.
This is a referential-integrity check.

In [ ]:
master_skus = set(sku_master['sku_id'])
sales_skus = set(sales_daily['sku_id'])
inventory_skus = set(inventory_snapshots['sku_id'])

print('SKUs in sku_master        :', sorted(master_skus))
print('SKUs in sales_daily       :', sorted(sales_skus))
print('SKUs in inventory_snapshots:', sorted(inventory_skus))
print()
print('All sales SKUs exist in master:', sales_skus.issubset(master_skus))
print('All inventory SKUs exist in master:', inventory_skus.issubset(master_skus))

## 9. Final Data Types After Cleaning

A quick summary of the cleaned data types before saving.

In [ ]:
print('--- sales_daily dtypes ---')
print(sales_daily.dtypes)
print()
print('--- sku_master dtypes ---')
print(sku_master.dtypes)
print()
print('--- inventory_snapshots dtypes ---')
print(inventory_snapshots.dtypes)

## 10. Save Cleaned Data to data/processed/

We save the cleaned DataFrames as new CSV files in `data/processed/`.
The raw files in `data/raw/` are **never modified**.

In [ ]:
processed_dir = Path('data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

sales_daily.to_csv(processed_dir / 'sales_daily_clean.csv', index=False)
sku_master.to_csv(processed_dir / 'sku_master_clean.csv', index=False)
inventory_snapshots.to_csv(processed_dir / 'inventory_snapshots_clean.csv', index=False)

print('Saved cleaned files to data/processed/:')
print('  sales_daily_clean.csv')
print('  sku_master_clean.csv')
print('  inventory_snapshots_clean.csv')

## 11. Verify the Saved Files

We read the saved files back to confirm they were written correctly.

In [ ]:
sales_check = pd.read_csv(processed_dir / 'sales_daily_clean.csv')
sku_check = pd.read_csv(processed_dir / 'sku_master_clean.csv')
inventory_check = pd.read_csv(processed_dir / 'inventory_snapshots_clean.csv')

print('Reloaded shapes:')
print('  sales_daily_clean         ->', sales_check.shape)
print('  sku_master_clean          ->', sku_check.shape)
print('  inventory_snapshots_clean ->', inventory_check.shape)
print()
print('--- sales_daily_clean (first 3 rows) ---')
print(sales_check.head(3))

## 12. Cleaning Summary

### What we cleaned
- **Date column**: converted from text (`str`) to proper **datetime** in both `sales_daily` and `inventory_snapshots`.
- **Text columns**: stripped whitespace and standardized casing — `sku_id` to uppercase, `product_name`/`category` to title case.

### What we checked (and confirmed)
- **No duplicate rows** in any dataset (0 before and after).
- **No missing values** in any dataset.
- **Stock formula** `closing_stock = opening_stock + units_received - units_sold` holds for all 42 inventory rows.
- **SKU consistency**: all SKUs in sales and inventory exist in `sku_master`.

### Output
- Cleaned files saved to `data/processed/`:
  - `sales_daily_clean.csv`
  - `sku_master_clean.csv`
  - `inventory_snapshots_clean.csv`
- The raw files in `data/raw/` were **not modified**.

The data is now clean and ready for the next step: **Exploratory Data Analysis (EDA)**.